In [ ]:
# Calcular tempo de execução
import time
tempo_inicio = time.time()

In [ ]:
!pip install Biopython pandas networkx matplotlib seaborn py3dmol scikit-learn plotly

### 0) Carga das bibliotecas

In [ ]:
from Bio.PDB import PDBList, PDBParser, MMCIFParser
from Bio.PDB.Polypeptide import is_aa
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from networkx.algorithms.community import louvain_communities
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.spatial import ConvexHull
import plotly.graph_objects as go
import plotly.io as pio

### 1) Preparaçãodos Dados da Proteína Selecionada

In [ ]:
# Proteína
# proteina = '1A6M'  # Mioglobina - 1 cadeia - 151 resíduos
# proteina = '1ZNJ'  # Insulina Humana - 12 Cadeias - 303 residuos
# proteina = '1WCM' # RNA Polimerase II - 12 cadeias - ~4.500 resíduos
proteina = '6B1T' # Adenovírus Humano - 

# Parâmetros
# Fazendo download da estrutura (mmCIF)
pdbl = PDBList()

arquivo = pdbl.retrieve_pdb_file(proteina, pdir=".", file_format="mmCif")  # Garante o formato correto
parser = MMCIFParser(QUIET=True) # Criando um objeto MMCIParser
    
# Declarando a estrutura
estrutura = parser.get_structure(proteina, arquivo)

### 2) Load e Parsing dos Dados da Proteína Selecionada

In [ ]:
# Reinicia os contadores
def total_reset(total):
    total = {
        'estrutura':len(estrutura),
        'modelo':0,
        'cadeia':0,
        'residuo':0,
        'atomo':0
    }
    return total

# Contablizar a quantidades de elementos
total = {}
total = total_reset(total)

# Coleção com informações sobre o ID (N°Cadeia_N°Residuo), Cadeia, N° Residuo, Aminoácido e as coordenadas de cada aminoácido.
estrutura_ca = []

# Reinicia os contadores
total = total_reset(total)

# Varre por toda estrutura e contabiliza cada elemento
for modelo in estrutura:
    total['modelo'] += 1
    for cadeia in modelo:
        total['cadeia'] += 1
        for residuo in cadeia: # Filtra apenas os aminoácidos (nós da rede)
            if (residuo.get_id()[0] == ' ' and is_aa(residuo, standard=True)) : # Se o valor for igual a vazio então é um aminoácido
                total['residuo'] += 1
                if ('CA' in residuo): # Pega os átomos que são apenas o Carbono Alfa
                    atomo = residuo['CA']
                    total['atomo'] += 1
                    id = cadeia.get_id() + '_' + str(residuo.get_id()[1])  # ID do Aminoácido na coleção
                    ca = {   # Dicionário contendo Id, Cadeia, N°Residuo, Aminácidos e Coordenadas
                            'id':id,
                            'cadeia': cadeia.get_id(),
                            'residuo_num': residuo.get_id()[1],
                            'aminoacido': residuo.get_resname(),
                            'x': atomo.get_coord()[0],
                            'y': atomo.get_coord()[1],
                            'z': atomo.get_coord()[2],
                         }  
                    estrutura_ca.append(ca) # Adiciona o aminoácido CA na lista de CAs (nós da rede)

print(f'Contabilizaçãoz: o algoritmo filtrou apenas os aminoácios CA')
print(f'Total de Estruturas: {total["estrutura"]}')
print(f'Total de Modelos: {total["modelo"]}')
print(f'Total de Cadeias: {total["cadeia"]}')
print(f'Total de Residuos: {total["residuo"]}')
print(f'Total de Átomos: {total["atomo"]}')
print(40 * '-')

# Converte a coleção (lista estrutura_ca) em um dataframe (Pandas)
df = pd.DataFrame(estrutura_ca)
df

### 3) Construção da Rede

In [ ]:
# Matriz das Coordenadas
df_coordenadas = df[['x','y','z']]
matriz_coordenadas = df_coordenadas.to_numpy()

# Tamanho da Rede (N) e Limiar para o Cálculo de Distância (R)
N = len(matriz_coordenadas)
R = 7.0

# Matriz Nula 
matriz_adjacencia = np.zeros((N,N), dtype=int)

In [ ]:
# Cálculo da Distância Espacial (dist)
for i in range(N):
    for j in range(i + 1, N):  #  Evite de explorar os autoloops 
       pontos_i = matriz_coordenadas[i]
       pontos_j = matriz_coordenadas[j]
       dist = np.linalg.norm(pontos_i - pontos_j)  # Cálculo da Distância Euclidiana
       if dist < R:
           # Como a matriz é simétrica as células [i][j] e [j][i] são preenchidas
           matriz_adjacencia[i][j] = 1   
           matriz_adjacencia[j][i] = 1

### 4) Análise Topológica da Rede

In [ ]:
# Instanciando um Grafo a partir da Matriz de Adjacência
G = nx.from_numpy_array(matriz_adjacencia)

# Dicionário com os Graus da rede
graus = dict(G.degree())

# Dicionário com a Centralidade de Intermediação da Rede
centralidade_intermediacao = nx.betweenness_centrality(G)

# Adicionando os valores de graus no DataFrame
df['grau'] = df.index.map(graus)

# Adicionando os valores de centralidade de intermediacao no DataFrame
df['c_intermediacao'] = df.index.map(centralidade_intermediacao)

In [ ]:
# Dataframe com as novas colunas grau e c_intermedicao
df

In [ ]:
# Top 10 (maiores) nós em n° de conexões (grau)
df[['aminoacido','grau','c_intermediacao']].sort_values(by='grau', ascending=False).head(10)

In [ ]:
# Configurar o estilo visual acadêmico
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5))

# Plotar a distribuição de graus (Histograma discreto)
sns.histplot(
    data=df,
    x="grau",
    stat="probability",  # Transforma o eixo Y em Probabilidade P(k)
    discrete=True,  # Ajusta as barras perfeitamente nos números inteiros
    color="skyblue",  # Mantém o tom azul suave que você usou
    edgecolor="black",  # Adiciona a borda preta nítida nas barras
)

# Customizar títulos e eixos
plt.title(f"Distribuição de Graus - {proteina}", fontsize=12, pad=10)
plt.xlabel("Grau (k)", fontsize=10)
plt.ylabel("P(k)", fontsize=10)

# Renderizar o gráfico na tela
plt.tight_layout()

# ==========================================
# Exportar a figura para usar no lattes
# ==========================================
plt.savefig("distribuicao_graus.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# Top 10 (maiores) nós que atuam como ponte (gatekeeper)
df[['aminoacido','grau','c_intermediacao']].sort_values(by='c_intermediacao', ascending=False).head(10)

In [ ]:
# Configurar o estilo visual acadêmico
sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5.5))

# Plotar a relação (Gráfico de Dispersão)
sns.scatterplot(
    data=df,
    x="grau",
    y="c_intermediacao",
    color="darkcyan",  # Cor única e sóbria para o preenchimento
    edgecolor="black",  # Borda preta nos pontos para destacar a sobreposição
    s=70,  # Tamanho dos pontos ajustado para legibilidade
    alpha=0.75,  # Opacidade para enxergar pontos sobrepostos
)

# Customizar títulos e eixos para o relatório
plt.title(
    f"Relação Topológica: Grau vs. Centralidade de Intermediação ({proteina})",
    fontsize=12,
    pad=12,
)
plt.xlabel("Grau (k) - Conectividade Local", fontsize=10)
plt.ylabel("Centralidade de Intermediação - Fluxo Global", fontsize=10)

# ==========================================
# Exportar a figura para usar no lattes
# ==========================================
plt.savefig("grau_vs_betweenness.png", dpi=300, bbox_inches="tight")

# 4. Renderizar o gráfico na tela
plt.tight_layout()
plt.show()

### 5) Detecção de Comunidades

In [ ]:
# Algoritmo de Louvain
comunidades = louvain_communities(G, seed=42)

# Dicionário de cuja chave é o nó e o valor a comunidade
dic_comunidades = {}

# Varre por cada comunidade definindo um id (inicia em zero)
for comunidade_id, comunidade in enumerate(comunidades):
    # Varre pelos nós em cada comunidade
    for no in comunidade:
        dic_comunidades[no] = comunidade_id # Adiciona o nó como chave e sua comunidade como valór no dicionário

# Adicionando os valores de comunidade no DataFrame
df['comunidade'] = df.index.map(dic_comunidades)

# Total de Comunidades
print(f"Total de comunidades detectadas: {len(comunidades)}")

In [ ]:
df

### 6) Validação Biológica

In [ ]:
# Gera um array com os ids da comunidades detectadas
comunidades_id = df['comunidade'].sort_values().unique()

# Varre por cada comunidade
for id in comunidades_id:
    # Agrupa as cadeias em um lista de cada comunidade
    lista_cadeias = list(df[df['comunidade'] == id]['cadeia'].unique())
    print(f'Comunidade: {id} e Cadeias Associadas: {lista_cadeias}')

In [ ]:
# Criar a tabela de contingência (Cadeias vs. Comunidades)
matriz_contingencia = pd.crosstab(df["cadeia"], df["comunidade"])

# Calcular Métricas de Validação Cruzada (Teoria da Informação)
# Comparamos o agrupamento real (cadeia) com o predito (comunidade)
ari = adjusted_rand_score(df["cadeia"], df["comunidade"])
nmi = normalized_mutual_info_score(df["cadeia"], df["comunidade"])

print(f"--- Métricas de Alinhamento Topologia-Biologia ---")
print(f"Índice de Rand Ajustado (ARI): {ari:.4f}")
print(f"Informação Mútua Normalizada (NMI): {nmi:.4f}")
print(40 * '-')

# Configurar a janela do gráfico (Ajustada para o volume de dados do 6B1T)
plt.figure(figsize=(16, 10))
sns.set_theme(style="white")

# Criamos uma máscara para esconder as células com valor 0, deixando o gráfico limpo
mask = matriz_contingencia == 0

# Plotar o Heatmap otimizado
sns.heatmap(
    matriz_contingencia,
    mask=mask,               # Aplica a máscara para omitir zeros
    annot=True,              # Exibe a quantidade exata de nós
    annot_kws={"size": 9},   # Tamanho da fonte dos números internos
    fmt="d",                 # Inteiros
    cmap="YlGnBu",           # Paleta amarelo-azul
    linewidths=0.5,          # Linha separadora fina
    linecolor="lightgray",
    cbar_kws={"label": "Quantidade de Nós (Aminoácidos)"},
)

# Customizar títulos e eixos para o relatório acadêmico
plt.title(
    f"Matriz de Validação: Cadeias Biológicas vs. Comunidades de Louvain ({proteina})\n"
    f"Alinhamento Topológico (NMI): {nmi:.2f}",
    fontsize=14,
    pad=15,
)
plt.xlabel("ID da Comunidade (Mapeamento Matemático)", fontsize=11)
plt.ylabel("Cadeia Estrutural do PDB (Entidade Biológica)", fontsize=11)

# Renderizar na tela
plt.tight_layout()

# ==========================================
# Exportar a figura para usar no lattes
# ==========================================
plt.savefig("matriz_validacao.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# Calcular o Centro Global da Proteína (Média de todas as coordenadas)
centro_global = df[['x', 'y', 'z']].mean().to_numpy()

# Agrupar por comunidade para calcular o centroide e a distância ao centro global
info_comunidades = []
for id_com, grupo in df.groupby('comunidade'):
    centroide = grupo[['x', 'y', 'z']].mean().to_numpy()
    dist_ao_centro = np.linalg.norm(centroide - centro_global)
    
    info_comunidades.append({
        'comunidade': id_com,
        'dist_centro': dist_ao_centro,
        'tamanho': len(grupo),
        'centroide_x': centroide[0],
        'centroide_y': centroide[1],
        'centroide_z': centroide[2]
    })

df_comunidades = pd.DataFrame(info_comunidades)

# Filtrar os 12 Pentons: São as 12 comunidades com maior distância ao centro global
pentons_meta = df_comunidades.nlargest(12, 'dist_centro')
ids_pentons = pentons_meta['comunidade'].tolist()

print(f"IDs das 12 comunidades identificadas como Pentons: {ids_pentons}")

In [ ]:
df_comunidades

In [ ]:
# Filtrar o DataFrame original para conter apenas os nós que pertencem aos 12 Pentons
df_pentons = df[df['comunidade'].isin(ids_pentons)]

# Descobrir quais cadeias biológicas exclusivas compõem esses Pentons
cadeias_dos_pentons = df_pentons['cadeia'].unique()

print(f"As cadeias biológicas que compõem os pentons neste modelo são: {cadeias_dos_pentons}")

In [ ]:
# Verifica quais cadeias estão em quais IDs de comunidades dos Pentons
for id_p in sorted(ids_pentons):
    cadeias_na_comunidade = df[df['comunidade'] == id_p]['cadeia'].unique()
    print(f"Comunidade {id_p} contém as Cadeias Biológicas: {list(cadeias_na_comunidade)}")

In [ ]:
# Extrair as coordenadas dos 12 centroides encontrados no seu código
pontos_pentons = pentons_meta[['centroide_x', 'centroide_y', 'centroide_z']].to_numpy()

# Calcular o Convex Hull (ele encontra automaticamente quais centroides se conectam 
# formando as faces triangulares do icosaedro)
hull = ConvexHull(pontos_pentons)
faces = hull.simplices  # Matriz com os índices dos triângulos (faces)

# Criar o gráfico de Malha 3D (Mesh3D) no Plotly
fig = go.Figure()

# Desenha o sólido translúcido (as faces do icosaedro)
fig.add_trace(go.Mesh3d(
    x=pontos_pentons[:, 0],
    y=pontos_pentons[:, 1],
    z=pontos_pentons[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    opacity=0.4,
    color='lightblue',
    name='Faces do Icosaedro'
))

# Desenhar as arestas pretas (as linhas dos triângulos)
for simprice in faces:
    # Fecha o triângulo conectando o ponto 0->1->2->0
    p_triangulo = pontos_pentons[np.append(simprice, simprice[0])]
    fig.add_trace(go.Scatter3d(
        x=p_triangulo[:, 0],
        y=p_triangulo[:, 1],
        z=p_triangulo[:, 2],
        mode='lines',
        line=dict(color='black', width=2),
        showlegend=False
    ))

# Adicionar os pontos dos Centroides (Vértices) em vermelho (Corrigido!)
fig.add_trace(go.Scatter3d(
    x=pontos_pentons[:, 0],
    y=pentons_meta['centroide_y'],  # <--- Corrigido para 'pentons_meta'
    z=pentons_meta['centroide_z'],  # <--- Corrigido para 'pentons_meta'
    mode='markers+text',
    text=pentons_meta['comunidade'].astype(str),  # Exibe o ID da comunidade
    textposition="top center",
    marker=dict(size=6, color='red'),
    name='Centroides (Pentons)'
))

# Ajustar Layout acadêmico
fig.update_layout(
    title="Triângulos dos centroides penton - Reconstrução do Icosaedro",
    scene=dict(
        xaxis=dict(backgroundcolor="white"),
        yaxis=dict(backgroundcolor="white"),
        zaxis=dict(backgroundcolor="white"),
        bgcolor="white"
    ),
    width=900, height=700
)

pio.renderers.default = "notebook"

fig.show()

In [ ]:
# Filtrar o DataFrame original para conter apenas os nós dos 12 Pentons
df_pentons = df[df['comunidade'].isin(ids_pentons)]

# Extrair os pontos dos centroides e calcular o Convex Hull para as faces
pontos_centroides = pentons_meta[['centroide_x', 'centroide_y', 'centroide_z']].to_numpy()
hull = ConvexHull(pontos_centroides)
faces = hull.simplices

# Inicializar a figura 3D
fig = go.Figure()

# --- CAMADA 1: As 12 Comunidades de Nós (Nuvens de Pontos) ---
# Lista de cores marcantes para diferenciar as 12 comunidades
cores_vivas = [
    'red', 'blue', 'green', 'orange', 'purple', 'cyan', 
    'magenta', 'yellow', 'brown', 'pink', 'lime', 'teal'
]

for i, id_penton in enumerate(ids_pentons):
    df_p = df_pentons[df_pentons['comunidade'] == id_penton]
    cor = cores_vivas[i]
    
    # Adiciona os aminoácidos/átomos daquela comunidade
    fig.add_trace(go.Scatter3d(
        x=df_p['x'], y=df_p['y'], z=df_p['z'],
        mode='markers',
        marker=dict(size=2.5, color=cor, opacity=0.4), # Opacidade leve para ver através
        name=f'Nós da Comunidade {id_penton}'
    ))

# --- CAMADA 2: O Sólido Geométrico (Faces Translúcidas) ---
fig.add_trace(go.Mesh3d(
    x=pontos_centroides[:, 0],
    y=pontos_centroides[:, 1],
    z=pontos_centroides[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    opacity=0.25, # Deixa bem transparente para as nuvens de pontos aparecerem dentro
    color='skyblue',
    name='Faces do Icosaedro'
))

# --- CAMADA 3: As Arestas dos Triângulos (Linhas Pretas) ---
for tri in faces:  # <--- Mudado de 'em' para 'in'
    p_triangulo = pontos_centroides[np.append(tri, tri[0])]
    fig.add_trace(go.Scatter3d(
        x=p_triangulo[:, 0], y=p_triangulo[:, 1], z=p_triangulo[:, 2],
        mode='lines',
        line=dict(color='black', width=3),
        showlegend=False
    ))

# --- CAMADA 4: Os Centroides (Esferas Vermelhas Grandes) - CORRIGIDO ---
fig.add_trace(go.Scatter3d(
    x=pontos_centroides[:, 0],
    y=pontos_centroides[:, 1],
    z=pontos_centroides[:, 2],
    mode='markers+text',
    text=pentons_meta['comunidade'].astype(str),
    textposition="top center",
    marker=dict(size=7, color='darkred', symbol='circle'), # <--- Alterado para 'circle'
    name='Centroides dos Pentons'
))

# --- CONFIGURAÇÕES DE LAYOUT ---
fig.update_layout(
    title="Validação Estrutural: Comunidades de Louvain e a Geometria Icosaédrica",
    scene=dict(
        xaxis=dict(title='X', backgroundcolor="white"),
        yaxis=dict(title='Y', backgroundcolor="white"),
        zaxis=dict(title='Z', backgroundcolor="white"),
        bgcolor="white"
    ),
    width=1100, height=850,
    showlegend=True
)

pio.renderers.default = "notebook"

# Renderizar o gráfico interativo
fig.show()

In [ ]:
# Recuperar os pontos dos 12 centroides e calcular o Convex Hull
pontos_pentons = pentons_meta[['centroide_x', 'centroide_y', 'centroide_z']].to_numpy()
hull = ConvexHull(pontos_pentons)

# Validação Geométrica (Elementos do Sólido)
num_vertices = len(pontos_pentons)
num_faces = len(hull.simplices)
# Arestas no Convex Hull podem ser extraídas diretamente dos 'simplices' ou calculadas por Euler: V - A + F = 2
num_arestas = num_vertices + num_faces - 2  

# Cálculo das Áreas de cada Face Triangular
# Usamos o produto vetorial para calcular a área de um triângulo em 3D formado por 3 pontos (A, B, C)
areas_triangulos = []
for face in hull.simplices:
    A = pontos_pentons[face[0]]
    B = pontos_pentons[face[1]]
    C = pontos_pentons[face[2]]
    
    # Vetores direcionais a partir do ponto A
    AB = B - A
    AC = C - A
    
    # A área do triângulo é metade da norma do produto vetorial entre AB e AC
    area = 0.5 * np.linalg.norm(np.cross(AB, AC))
    areas_triangulos.append(area)

areas_triangulos = np.array(areas_triangulos)

# Métricas Estatísticas da Variação das Áreas
media_areas = np.mean(areas_triangulos)
desvio_padrao_areas = np.std(areas_triangulos)
coeficiente_variacao = desvio_padrao_areas / media_areas
coeficiente_variacao_pct = coeficiente_variacao * 100

# Imprimir a Saída Formatada exatamente igual ao print enviado
print("----- Validação geométrica -----")
print(f"Vértices: {num_vertices}")
print(f"Faces triangulares: {num_faces}")
print(f"Arestas: {num_arestas}")
print("\nEstrutura compatível com um icosaedro.")
print("\n----- Variação das áreas -----")
print(f"Média das áreas: {media_areas:.4f}")
print(f"Desvio padrão das áreas do triangulo: {desvio_padrao_areas:.4f}")
print(f"Coeficiente de variação: {coeficiente_variacao:.4f}")
print(f"Coeficiente de variação (%): {coeficiente_variacao_pct:.2f}%")

In [ ]:
tempo_final = time.time()
duracao = tempo_final - tempo_inicio

print(f"Tempo total de processamento: {int(duracao // 60)}m {int(duracao % 60)}s")